# Day 8：亲眼看懂RMSNorm与RoPE

先读 [Day 8图解](../docs/DAY08_RMSNORM_ROPE.md) 和 [学习记录](../docs/DAY08_STUDY_LOG.md)。
RMSNorm像调节每一行信号音量；RoPE像按位置转动Q/K特征对。
CPU FP64公式oracle与FP32/低精度参考验证，不加载Qwen3权重，不测GPU速度。
Mini RMSNorm内部明确使用FP32，即使.double()也不升级内部归约精度；容差按算法精度设定。
先写预测，另建自己的实现，再与参考对齐；运行本Notebook不等于已独立实现。

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/mini_transformer/model.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import torch
from learning.mini_transformer.model import RMSNorm, rope
from reasoning_from_scratch.qwen3 import RMSNorm as RepositoryRMSNorm
from reasoning_from_scratch.qwen3 import apply_rope, compute_rope_params

## 1. RMSNorm：先手算，再对齐

输入`[3,4]`，先预测`mean(x**2)`、RMS和不带gamma的输出。
RMSNorm不减均值；`gamma`这里只用全1。
RMSNorm=Root Mean Square Layer Normalization，RoPE=Rotary Position Embedding。
完整前向推导、L2区别和一手链接见 [RMSNorm公式推导](../docs/DAY08_RMSNORM_ROPE.md#rmsnorm-derivation)。
先从希望mean((c*x)**2)=1推得c=1/sqrt(mean(x**2))，再加入eps和每特征gamma。
先用FP64公式手算，再与Mini的FP32内部计算作合理容差比较，不将FP64输入误当高精度实现。

### 1.1 前向公式推导：为什么除以均方根？

**目标不是把每个数变成1，而是调节一整行特征的共同尺度。**
只看一个位置的输入，E是特征数；对[B,T,E]沿最后一维计算，不混入其他token。

$$x=[x_1,x_2,\ldots,x_E]$$

先平方避免正负抵消，再平均、开平方，得到Root Mean Square（均方根）：

$$q=\frac{1}{E}\sum_{j=1}^{E}x_j^2,\qquad \operatorname{RMS}(x)=\sqrt{q}$$

先忽略eps和gamma，对非零向量设一个正缩放系数c，令新向量的均方根为1：

$$\hat{x}_i=cx_i,\qquad \frac{1}{E}\sum_{i=1}^{E}(cx_i)^2=1$$

把共同的c平方提出来，再取正解：

$$c^2\frac{1}{E}\sum_{i=1}^{E}x_i^2=1$$

$$c=\frac{1}{\sqrt{\frac{1}{E}\sum_{i=1}^{E}x_i^2}}=\frac{1}{\operatorname{RMS}(x)}$$

所以共同缩放就是除以RMS，不是各分量分别除以自己：

$$\hat{x}_i=\frac{x_i}{\operatorname{RMS}(x)}$$

这是选定归一化目标后的推导，不是证明所有Norm都必须使用同一种目标。

### 1.2 加入epsilon和gamma，得到实际RMSNorm

零向量的RMS为0，因此在平方根内加入正稳定项epsilon；再按特征乘可学习增益gamma：

$$\boxed{y_i=\gamma_i\frac{x_i}{\sqrt{\frac{1}{E}\sum_{j=1}^{E}x_j^2+\epsilon}}}$$

| 符号 | 意义 | 代码位置 |
| --- | --- | --- |
| E | 当前归一化行的特征数 | x.shape[-1] |
| q | 平方均值，不是先减均值的方差 | x.square().mean(dim=-1,keepdim=True) |
| epsilon / ε | 分母稳定项，epsilon读艾普西龙 | eps，本课显式1e-6 |
| gamma / γ | 可学习逐特征增益，gamma读伽马 | weight，shape [E] |

gamma全为1时，加入eps后的均方根为：

$$\operatorname{RMS}(\hat{x})=\sqrt{\frac{q}{q+\epsilon}}$$

q远大于eps时才接近1；零向量输出0。再乘不同gamma后更不保证RMS为1。
**RMSNorm不减均值，不要求输出和为1，也不保证零均值。**

对应本课程代码路径：

```python
input_dtype = x.dtype
x = x.float()
mean_square = x.square().mean(dim=-1, keepdim=True)
normalized = x * torch.rsqrt(mean_square + eps)
y = (normalized * weight).to(input_dtype)
```

rsqrt(a)=1/sqrt(a)。keepdim=True把平方均值保留为[B,T,1]，便于广播；weight应注册为可学习参数。
Mini内部固定FP32，即使.double()也不升级这步精度；后续用FP64公式oracle和适合FP32的容差验证。

### 1.3 手算[3,4]：为什么不是[0.6,0.8]？

忽略eps、gamma为[1,1]时：

```text
平方均值 = (3²+4²)/2 = 12.5
RMS = sqrt(12.5) ≈ 3.5355
输出 ≈ [3/3.5355,4/3.5355] = [0.8485,1.1314]
```

L2归一化除的是向量长度5，得到[0.6,0.8]；两者目标不同：

$$\lVert x\rVert_2=\sqrt{\sum_i x_i^2},\qquad \operatorname{RMS}(x)=\frac{\lVert x\rVert_2}{\sqrt{E}}$$

| 方法 | 被归一到1的量，忽略eps | [3,4]的输出 |
| --- | --- | --- |
| L2归一化 | 二范数 | [0.6,0.8] |
| RMS归一化，gamma=1 | 均方根 | [0.8485,1.1314] |

RMS归一化后的二范数是sqrt(2)，不是1；不是哪个算错了。

**一手阅读与源码链接：**

- [PyTorch 2.10 RMSNorm：公式、eps和参数](https://docs.pytorch.org/docs/2.10/generated/torch.nn.RMSNorm.html)
- [RMSNorm原论文：Zhang与Sennrich，2019](https://arxiv.org/abs/1910.07467)
- [PyTorch 2.10 rsqrt定义](https://docs.pytorch.org/docs/2.10/generated/torch.rsqrt.html)
- [Mini RMSNorm源码](mini_transformer/model.py)
- [本地实战接口与验收教程](../docs/LOCAL_PRACTICE.md)

下面执行原有手算与边界实验；本节仅推导前向，不等于完整反向推导或性能证明。

In [ ]:
x = torch.tensor([[3., 4.]], dtype=torch.float64)
eps = 1e-6
manual_rms = torch.sqrt(x.square().mean(-1, keepdim=True) + eps)
manual = x / manual_rms
print('mean(x^2):', x.square().mean().item())
print('RMS:', manual_rms.item())
print('RMSNorm(x):', manual.tolist())
assert manual.shape == x.shape
assert manual.mean().item() > 0
norm = RMSNorm(2)
actual = norm(x.float()).detach().double()
torch.testing.assert_close(actual, manual, atol=1e-6, rtol=1e-5)
print('Mini FP32最大误差:', (actual - manual).abs().max().item())

### 边界输入：不要只测随机正态

全零依靠eps保持有限；常数向量不变成零中心；大/小值仍应有限。
独立PyTorch公式作为oracle，不只比较字符串或打印结果。

In [ ]:
boundary = torch.tensor([[0., 0., 0.], [2., 2., 2.], [1e4, -1e4, 1e4], [1e-4, -2e-4, 3e-4]], dtype=torch.float64)
norm3 = RMSNorm(3)
expected = boundary / torch.sqrt(boundary.square().mean(-1, keepdim=True) + eps)
got = norm3(boundary.float()).detach().double()
torch.testing.assert_close(got, expected, atol=1e-6, rtol=1e-5)
assert torch.isfinite(got).all()
print('input:', boundary.tolist())
print('output:', got.tolist())
print('constant row mean:', got[1].mean().item(), '(RMSNorm不是零中心LayerNorm)')
print('边界最大误差:', (got - expected).abs().max().item())

### 非全1的gamma、低精度和梯度

固定eps=1e-6、gamma=[0.5,1,1.5]，分别测试FP32/FP16/BF16，oracle由输入的实际量化值转FP64计算。
对照Mini、仓库RMSNorm与torch.nn.RMSNorm；不使用后者的默认eps。
gamma是逐特征增益，输出不要求零均值或严格单位RMS。CPU测试不证明GPU性能。

In [ ]:
gamma = torch.tensor([0.5, 1., 1.5])
for dtype, atol, rtol in ((torch.float32, 1e-6, 1e-5),
                         (torch.float16, 2e-3, 2e-3),
                         (torch.bfloat16, 2e-2, 1e-2)):
    sample = boundary.to(dtype)
    oracle_input = sample.double()
    oracle = oracle_input / torch.sqrt(oracle_input.square().mean(-1, keepdim=True) + eps)
    oracle = oracle * gamma.double()
    mini = RMSNorm(3, eps=eps)
    repo = RepositoryRMSNorm(3, eps=eps)
    builtin = torch.nn.RMSNorm(3, eps=eps, dtype=dtype)
    with torch.no_grad():
        mini.weight.copy_(gamma)
        repo.scale.copy_(gamma)
        builtin.weight.copy_(gamma)
    for name, implementation in (('mini', mini), ('repository', repo), ('torch', builtin)):
        output = implementation(sample)
        assert output.dtype == dtype and torch.isfinite(output).all()
        torch.testing.assert_close(output.double(), oracle, atol=atol, rtol=rtol)
        print(dtype, name, 'max_error=', (output.double() - oracle).abs().max().item())
train_x = torch.tensor([[[1., 2., 3.], [-2., 1., 4.]]], requires_grad=True)
train_norm = RMSNorm(3)
train_norm(train_x).square().mean().backward()
assert torch.isfinite(train_x.grad).all() and torch.isfinite(train_norm.weight.grad).all()
print('输入/gamma梯度存在且有限：passed')

## 2. RoPE：位置改变角度，shape不变

创建`[B,H,T,D]=[1,2,3,4]`，比较offset=0和offset=5。
先预测：两者shape是否相同？数值是否相同？D为4意味着每个head有2个旋转对。
本仓库采用split-halves：配对(0,2)、(1,3)，不是相邻配对。theta明确统一为10000。

### 2.1 配对、旋转角与计算写法，是三个不同问题

**RoPE本质就是二维旋转。split-halves与interleaved都要旋转，不是只有另一种才按theta旋转。**

| 问题 | 含义 | 本例 |
| --- | --- | --- |
| 谁和谁一对 | 特征布局 | split-halves配(0,2)/(1,3)，interleaved配(0,1)/(2,3) |
| 转多少 | 位置与每对频率 | phi=position*frequency |
| 怎样算 | 等价计算表达 | sin/cos、二维旋转矩阵或同一对的复数乘法 |

对选定的一对(a,b)：

$$\begin{bmatrix}a'\\b'\end{bmatrix}=\begin{bmatrix}\cos\phi&-\sin\phi\\\sin\phi&\cos\phi\end{bmatrix}\begin{bmatrix}a\\b\end{bmatrix}$$

展开为`a'=a*cos(phi)-b*sin(phi)`、`b'=a*sin(phi)+b*cos(phi)`。
同一对写成复数z=a+ib时，可用`z'=z*exp(i*phi)`实现；这不自动解决特征布局差异。

**本接口的theta=10000是频率基数，不是直接旋转10000弧度。**

$$\omega_i=\mathrm{theta\_base}^{-2i/D},\qquad \phi_{p,i}=p\omega_i$$

i从0到D/2-1。D=4、基数10000、位置p=2时：

```text
第0对：频率1    → 角度2弧度
第1对：频率0.01 → 角度0.02弧度
```

split-halves把两角应用到(x0,x2)/(x1,x3)，interleaved应用到(x0,x1)/(x2,x3)。
不同配对直接作用于同一原始排列通常得到不同结果；切换时需要对应特征与投影权重的一致转换。
本课验收固定split-halves，不改变my_ops.py接口，也不改为其他长上下文缩放方案。

深入链接：[RoFormer原论文](https://arxiv.org/abs/2104.09864)、
[Llama 3官方复数旋转实现](https://github.com/meta-llama/llama3/blob/main/llama/model.py)、
[本仓库两种配对示意](../reasoning_from_scratch/qwen3.py)、
[完整中文说明](../docs/DAY08_RMSNORM_ROPE.md#rope-angle-layout)。

In [ ]:
generator = torch.Generator().manual_seed(8)
q = torch.randn(1, 2, 3, 4, dtype=torch.float32, generator=generator)
theta = 10_000.0
rot0 = rope(q, start=0, theta=theta)
rot5 = rope(q, start=5, theta=theta)
assert rot0.shape == rot5.shape == q.shape
assert not torch.allclose(rot0, rot5)
cos, sin = compute_rope_params(4, theta_base=theta, context_length=16)
ref0 = apply_rope(q, cos, sin, 0)
ref5 = apply_rope(q, cos, sin, 5)
torch.testing.assert_close(rot0, ref0, atol=1e-6, rtol=1e-6)
torch.testing.assert_close(rot5, ref5, atol=1e-6, rtol=1e-6)
print('input shape:', tuple(q.shape))
print('offset0/5 shape:', tuple(rot0.shape), tuple(rot5.shape))
print('max difference:', (rot0-rot5).abs().max().item())
torch.testing.assert_close(rot5.norm(dim=-1), q.norm(dim=-1), atol=1e-6, rtol=1e-5)
torch.testing.assert_close(rot0[:, :, :1], q[:, :, :1], atol=0, rtol=0)
print('每头二范数保持、位置0不旋转：passed')

### 一整段与分块旋转必须一致；同内容的相对位置点积

先预测：同一个7位置输入，前3位置start=0、后4位置start=3是否等于整段start=0？
再固定两个内容向量，比较位置2/5与位置7/10，位置差都是3，点积应在容差内一致。
这不是说任意不同token只要距离相同就有相同分数。

In [ ]:
sequence = torch.randn(2, 2, 7, 4, generator=generator)
full = rope(sequence, start=0, theta=theta)
parts = torch.cat((rope(sequence[:, :, :3], start=0, theta=theta),
                   rope(sequence[:, :, 3:], start=3, theta=theta)), dim=2)
torch.testing.assert_close(parts, full, atol=1e-6, rtol=1e-6)
wrong = torch.cat((rope(sequence[:, :, :3], start=0, theta=theta),
                   rope(sequence[:, :, 3:], start=0, theta=theta)), dim=2)
assert not torch.allclose(wrong, full)
fixed_q = torch.tensor([[[[1., 2., 3., 4.]]]])
fixed_k = torch.tensor([[[[4., -1., 2., 3.]]]])
score_2_5 = (rope(fixed_q, 2, theta) * rope(fixed_k, 5, theta)).sum(-1)
score_7_10 = (rope(fixed_q, 7, theta) * rope(fixed_k, 10, theta)).sum(-1)
torch.testing.assert_close(score_2_5, score_7_10, atol=1e-5, rtol=1e-5)
print('整段/正确split最大误差:', (parts - full).abs().max().item())
print('错误重置start的最大误差:', (wrong - full).abs().max().item())
print('相同内容、相同位置差:', score_2_5.tolist(), score_7_10.tolist())

### RoPE负例与位置解释

head_dim必须是正偶数；offset=5只改变位置范围`[5,6,7]`，不增加维度。
RoPE不产生causal mask，未来是否可见仍由mask决定。

In [ ]:
bad = torch.randn(1, 1, 2, 3, generator=generator)
try:
    rope(bad)
except ValueError as error:
    print('expected odd-D rejection:', error)
else:
    raise AssertionError('Odd head dimension must be rejected.')
try:
    rope(torch.randn(1, 1, 2, 4, generator=generator), start=-1)
except ValueError as error:
    print('expected negative-offset rejection:', error)
else:
    raise AssertionError('Negative offset must be rejected.')

## 自测与参考答案

**1. RMSNorm会减均值吗？** 不会；只按最后特征维度计算平方均值并缩放。

**2. offset从0改5为什么shape不变？** B/H/T/D没变，只是使用了不同位置的旋转角度。

**3. RoPE负责屏蔽未来吗？** 不负责；RoPE注入位置，causal mask限制可见性。

**4. 为什么只旋转Q/K？** 本课程实现让位置影响匹配分数，V仍作为被汇集内容；以目标模型源码为准。

**5. 通过随机样例就够了吗？** 不够，要有边界输入、奇数D/负offset负例、不同offset、dtype和独立oracle。

**6. split-halves的4维怎么配对？** (0,2)、(1,3)。与相邻配对不是同一数组排列，不能无声混用。

**7. 为什么FP64输入也不能要求Mini RMSNorm达到1e-12？** 内部明确转FP32，.double()不改变这条计算路径。

**8. 今天完成独立实现了吗？** 本Notebook只提供已验证参考；学习者应另建自己的实现并记录预测、数值对齐与失败用例。

个人结果记入 [Day 8学习记录](../docs/DAY08_STUDY_LOG.md)，不要用参考输出自动勾选。